# Context-Scoped Cortex Agent Requests

This notebook demonstrates how to scope a Cortex Agent's responses to the user's current context by injecting page/user state into the **system prompt** (`instructions.response` field) of the REST API request.

## The Pattern

```
┌─────────────────────────────┐
│  Application Layer          │
│                             │
│  User is viewing:           │
│  • Executive Dashboard      │
│  • MRR: $2.44M             │
│  • Churn: 3.8%             │
│                             │
│  User asks: "What should    │
│  I focus on?"              │
└──────────────┬──────────────┘
               │
               ▼
┌─────────────────────────────┐
│  API Request                │
│                             │
│  instructions.response:     │
│    "You are an AI assistant │
│     ...                     │
│     Current context:        │
│     - Page: Exec Dashboard  │
│     - MRR: $2.44M          │
│     - Churn: 3.8% (rising) │
│     ..."                    │
│                             │
│  messages:                  │
│    [{role: user,            │
│      text: "What should I   │
│      focus on?"}]           │
└──────────────┬──────────────┘
               │
               ▼
┌─────────────────────────────┐
│  Cortex Agent               │
│                             │
│  Responds with churn-       │
│  focused analysis because   │
│  it knows that's what the   │
│  user is looking at.        │
└─────────────────────────────┘
```

The key insight: **the same question produces different, more relevant answers** when the agent knows what the user is currently viewing.

## Setup

Before running this notebook:
1. Run `setup.sql` against your Snowflake account to create the demo data
2. Set environment variables (or modify the connection params below)

In [1]:
import json
import requests

# Connection name from ~/.snowflake/config.toml
CONNECTION_NAME = "parker_demo"

# Agent object location
AGENT_DATABASE = "CORTEX_AGENT_DEMO"
AGENT_SCHEMA = "ANALYTICS"
AGENT_NAME = "ANALYTICS_AGENT"

In [2]:
# Authenticate using PAT from connections.toml
import tomllib
from pathlib import Path

config_path = Path.home() / ".snowflake" / "config.toml"
with open(config_path, "rb") as f:
    config = tomllib.load(f)

connection = config["connections"][CONNECTION_NAME]
ACCOUNT = connection["account"]
PAT = connection["password"]  # This is a Programmatic Access Token (JWT)

# Snowflake hostnames use hyphens, not underscores
HOST = ACCOUNT.lower().replace("_", "-") + ".snowflakecomputing.com"
API_URL = f"https://{HOST}/api/v2/databases/{AGENT_DATABASE}/schemas/{AGENT_SCHEMA}/agents/{AGENT_NAME}:run"

HEADERS = {
    "Authorization": f"Bearer {PAT}",
    "Content-Type": "application/json",
    "Accept": "application/json",
}

print(f"Using PAT auth. Agent endpoint: {API_URL}")

Using PAT auth. Agent endpoint: https://sfsenorthamerica-perickson-aws1.snowflakecomputing.com/api/v2/databases/CORTEX_AGENT_DEMO/schemas/ANALYTICS/agents/ANALYTICS_AGENT:run


## Helper Function

This function builds the API payload with context injected into the system prompt.

In [3]:
def ask_agent(user_question: str, page_context: dict, debug: bool = False) -> str:
    """Send a context-scoped question to the named agent."""
    context_text = "\n".join(f"- {k}: {v}" for k, v in page_context.items())
    
    payload = {
        "messages": [
            {
                "role": "system",
                "content": [{"type": "text", "text": f"Here is my current application context:\n{context_text}"}],
            },
            {
                "role": "user",
                "content": [{"type": "text", "text": user_question}],
            },
        ],
        "stream": False,
    }
    
    if debug:
        print("=== REQUEST PAYLOAD ===")
        print(json.dumps(payload, indent=2)[:2000])
        print("=======================\n")
    
    resp = requests.post(API_URL, headers=HEADERS, json=payload, timeout=120)
    
    if not resp.ok:
        print(f"=== ERROR {resp.status_code} ===")
        print(resp.text[:2000])
        print("========================")
        resp.raise_for_status()
    
    data = resp.json()
    
    if debug:
        print("=== RAW RESPONSE ===")
        print(json.dumps(data, indent=2)[:3000])
        print("====================\n")
    
    # Non-streaming response: top-level content array
    texts = [item["text"] for item in data.get("content", []) if item.get("type") == "text"]
    return "\n".join(texts) if texts else "(No text response)"

## Demo: Same Question, Different Contexts

We'll ask the same question — **"What should I focus on?"** — but scope it to three different pages. Watch how the agent's response changes based on context.

### Context 1: Executive Dashboard

The user is a VP of Growth looking at high-level KPIs. Churn is trending up.

In [4]:
dashboard_context = {
    "Page": "Executive Dashboard",
    "Time Period": "Last 30 days (December 2024)",
    "Visible KPIs": "MRR $2.44M (+2.5% MoM), Churn Rate 3.8% (up from 3.6%), Active Users 18,200, NRR 101.2%",
    "Trend Alert": "Churn rate has increased for 6 consecutive months (2.1% → 3.8%)",
    "User Role": "VP of Growth",
}

In [5]:
response_dashboard = ask_agent("What should I focus on?", dashboard_context, debug=True)
print("=== AGENT RESPONSE (Executive Dashboard context) ===")
print(response_dashboard)

=== REQUEST PAYLOAD ===
{
  "messages": [
    {
      "role": "system",
      "content": [
        {
          "type": "text",
          "text": "Here is my current application context:\n- Page: Executive Dashboard\n- Time Period: Last 30 days (December 2024)\n- Visible KPIs: MRR $2.44M (+2.5% MoM), Churn Rate 3.8% (up from 3.6%), Active Users 18,200, NRR 101.2%\n- Trend Alert: Churn rate has increased for 6 consecutive months (2.1% \u2192 3.8%)\n- User Role: VP of Growth"
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": "What should I focus on?"
        }
      ]
    }
  ],
  "stream": false
}

=== RAW RESPONSE ===
{
  "content": [
    {
      "tool_use": {
        "client_side_execute": false,
        "input": {
          "pruning_question": "What are the key metrics and areas to focus on for business performance?"
        },
        "name": "analyst_tool",
        "tool_use_id": "toolu_bdrk_01RvWLEQSSGEuwSG

### Context 2: Pipeline & Funnel

The user is the Head of Sales, looking at Enterprise conversion funnel.

In [6]:
pipeline_context = {
    "Page": "Pipeline & Funnel Analysis",
    "Time Period": "December 2024",
    "Selected Segment": "Enterprise",
    "Funnel Summary": "320 Awareness → 210 Interest (65.6%) → 95 Evaluation (45.2%) → 42 Negotiation (44.2%) → 28 Closed Won (66.7%)",
    "Avg Deal Size": "$92,000 (Closed Won)",
    "Comparison": "vs November: Closed Won up from 24 to 28 (+16.7%), conversion from Negotiation improved from 63.2% to 66.7%",
    "User Role": "Head of Sales",
}

response_pipeline = ask_agent("What should I focus on?", pipeline_context)
print("=== AGENT RESPONSE (Pipeline & Funnel context) ===")
print(response_pipeline)

=== AGENT RESPONSE (Pipeline & Funnel context) ===


Based on your current business data, here are the **three critical areas** that need your immediate attention:

## 1. **Rising Churn & Declining Net Revenue Retention**

Your most recent metrics show a concerning trend:




- Churn rate has increased from 3.50% to 3.80% over the last three months
- Net Revenue Retention declined from 102.50% to 101.20%
- While you're adding new customers (68 in December vs 58 in October), the increasing churn is offsetting growth

## 2. **SMB Segment Health Crisis**

Your SMB segments represent **1,680 customers** (72% of your customer base) but are showing the weakest performance:




**Critical issues in SMB:**
- **SMB APAC**: 81.50% retention (lowest across all segments), only 61.80% monthly active rate
- **SMB EMEA**: 83.10% retention, 64.20% monthly active rate
- Low NPS scores (54-58) indicate dissatisfaction
- Minimal expansion revenue (3.2-4.2%) suggests low product value realization

## 3. *

### Context 3: Customer Segments

The user is a Customer Success Manager, viewing the Enterprise US West segment.

In [7]:
segment_context = {
    "Page": "Customer Segments Detail View",
    "Selected Segment": "Enterprise US West",
    "Segment Metrics": "85 customers, Avg deal $95K, Retention 96.2%, NPS 72.5, Monthly active 88.3%",
    "Top Feature": "Advanced Analytics",
    "Expansion Revenue": "12.4% (highest across all segments)",
    "Support Load": "2.1 tickets/customer/month (below average)",
    "User Role": "Customer Success Manager",
}

response_segment = ask_agent("What should I focus on?", segment_context)
print("=== AGENT RESPONSE (Customer Segments context) ===")
print(response_segment)

=== AGENT RESPONSE (Customer Segments context) ===


Based on your data, here are the **three critical areas** you should focus on:

## 1. **Rising Churn Rate** 🚨 (Most Urgent)

Your churn rate has increased from 3.0% to 3.8% over the last six months. While your MRR is growing ($2.15M → $2.44M), the accelerating churn threatens this growth. 




The chart shows churn rising steadily while Net Revenue Retention (NRR) is declining from 104.2% to 101.2%, indicating that expansion revenue is also slowing.

## 2. **SMB Segment Retention** (Highest Impact)

Your SMB segments have significantly lower retention rates, especially in APAC:




**Key insights:**
- **SMB APAC**: 81.5% retention, 54.2 NPS, only 61.8% monthly active rate
- **SMB EMEA**: 83.1% retention, 55.8 NPS  
- **SMB US**: 85.4% retention (better, but still concerning)

Compare this to Enterprise US West at 96.2% retention. SMB represents 1,680 customers (73% of your customer base), so even small retention improvements here wil

### Contrast: No Context (Generic Response)

Without any page context, the agent gives a generic answer.

In [8]:
# Send the same question with NO page context — just a plain message to the agent
no_context_payload = {
    "messages": [
        {
            "role": "user",
            "content": [{"type": "text", "text": "What should I focus on?"}],
        }
    ],
    "stream": False,
}

resp = requests.post(API_URL, headers=HEADERS, json=no_context_payload, timeout=120)
resp.raise_for_status()
data = resp.json()
texts = [item["text"] for item in data.get("content", []) if item.get("type") == "text"]

print("=== AGENT RESPONSE (NO context) ===")
print("\n".join(texts) if texts else "(No response)")

=== AGENT RESPONSE (NO context) ===


Based on your current data, here are the **top 3 areas you should focus on**:

## 1. **Rising Churn Rate - Immediate Priority** 🚨

Your churn rate has increased **27% over the past 6 months** (from 3.00% in July to 3.80% in December 2024). At the same time, your Net Revenue Retention has declined from 104.20% to 101.20%.




**Action Items:**
- Investigate what's driving the increased churn in recent months
- Focus retention efforts on at-risk segments (see #2 below)

## 2. **SMB Segment Performance - High Risk** ⚠️

Your SMB segments, particularly **SMB APAC**, are significantly underperforming:




**Key concerns:**
- **SMB APAC**: 81.5% retention (lowest), 54.2 NPS, only 61.8% monthly engagement
- **SMB EMEA**: 83.1% retention, 55.8 NPS
- All SMB segments have low expansion revenue (3-4%) vs. Enterprise (9-12%)

**Action Items:**
- Prioritize customer success resources for SMB segments, especially APAC
- Investigate why SMB engagement is low (61

## Key Takeaways

1. **Agent object owns config** — tools, tool_resources, model, and base instructions are defined once via `CREATE AGENT`
2. **Messages carry the context** — dynamic page/user state is prepended to the user message at runtime
3. **Same agent, same tools, same data** — only the message context changes per request
4. **Context can include anything**: current page, visible metrics, user role, selected filters, time range, etc.

### Production Considerations

- **Dynamic context**: In a real app, pull context from your app state (Redux store, URL params, DB session)
- **Context size**: Keep it focused — include what's visible, not everything in the database
- **User permissions**: Context can also encode what the user is *allowed* to see
- **Threads**: Use `thread_id` + `parent_message_id` to maintain conversation history server-side